# 01 — Dane

**Cel notebooka:** pobrać z [OpenF1](https://openf1.org/docs) wszystko, czego potrzebujemy do przewidzenia wyniku GP Singapuru 2026, i zapisać to na dysk — kolejne notebooki tylko wczytują pliki CSV.

| Plik | Zawartość | Po co |
|---|---|---|
| `races_2026.csv` | lista wyścigów 2026 z numerem rundy | oś czasu |
| `results_2026.csv` | wyniki wyścigów | **zmienne docelowe** (wygrana, top 3, top 10) + forma |
| `quali_2026.csv` | wyniki kwalifikacji, także Singapur | pozycja startowa, tempo |
| `sprint_2026.csv` | wyniki sprintów, także Singapur | dodatkowa informacja o tempie w weekendzie |
| `fastest_laps_2026.csv` | kto miał najszybsze okrążenie | **zmienna docelowa** nr 4 |
| `teammate_gaps_hist.csv` | różnica w kwalifikacjach do partnera z zespołu, 2023–2025 | **umiejętności kierowcy** niezależne od samochodu |

Każdy krok ma ten sam układ: *co i dlaczego* → kod → *co tu się dzieje*.

> Pierwsze uruchomienie trwa kilka minut (limit zapytań API). Kolejne — sekundy, bo wszystko czytamy z `data/raw`.

## 1.1 Funkcja do pobierania danych

**Co robimy:** jedną funkcję `get()`, przez którą przechodzi każde zapytanie do OpenF1.

**Dlaczego:** będziemy pobierać dane z ponad stu sesji. Bez zapisu na dysk każde ponowne uruchomienie notebooka ściągałoby wszystko od nowa — wolno i obciążając API. Funkcja przy pierwszym wywołaniu pobiera dane i zapisuje JSON, przy kolejnych czyta z pliku.

In [ ]:
import json, time
from pathlib import Path
import requests
import pandas as pd

BASE_URL = "https://api.openf1.org/v1"
CACHE_DIR = Path("data/raw")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def get(endpoint, **params):
    """Pobiera dane z OpenF1; odpowiedź zapisuje na dysk i przy kolejnym wywołaniu czyta z pliku."""
    key = endpoint + "_" + "_".join(f"{k}={v}" for k, v in sorted(params.items()))
    path = CACHE_DIR / f"{key}.json"
    if path.exists():
        return pd.DataFrame(json.loads(path.read_text()))
    for attempt in range(5):
        resp = requests.get(f"{BASE_URL}/{endpoint}", params=params, timeout=30)
        if resp.status_code != 429:
            break
        wait = 5 * 2 ** attempt
        print(f"429 (za dużo zapytań) — czekam {wait} s...")
        time.sleep(wait)
    time.sleep(1)
    if resp.status_code == 404:  # OpenF1 zwraca 404 "No results found", gdy danych (jeszcze) nie ma
        return pd.DataFrame()
    resp.raise_for_status()
    data = resp.json()
    path.write_text(json.dumps(data))
    return pd.DataFrame(data)

**Co tu się dzieje:**
- `**params` zbiera argumenty nazwane: `get("sessions", year=2026)` → zapytanie `.../sessions?year=2026`.
- `key` to nazwa pliku z endpointu i parametrów, np. `sessions_year=2026.json`. Parametry są posortowane, więc ta sama kombinacja zawsze trafia do tego samego pliku.
- **Pętla `for attempt`** — API ma limit zapytań. Po przekroczeniu odpowiada kodem **429 Too Many Requests** (przekonaliśmy się przy pierwszym uruchomieniu!). Czekamy i próbujemy ponownie, za każdym razem 2× dłużej: 5, 10, 20, 40, 80 s. To standardowy wzorzec *exponential backoff*.
- **404** — OpenF1 tak sygnalizuje „brak danych” (np. wynik wyścigu, który jeszcze się nie odbył). Zwracamy pustą tabelę i **nie zapisujemy jej na dysk** — inaczej po wyścigu dalej czytalibyśmy pusty plik.
- `raise_for_status()` rzuca błąd przy pozostałych problemach (500 itp.).
- ⚠️ Gdy dane na serwerze się zmienią, trzeba usunąć odpowiedni plik z `data/raw`, żeby pobrać świeżą wersję.

## 1.2 Kalendarz sezonu 2026

**Co robimy:** pobieramy wszystkie sesje 2026, zamieniamy daty z tekstu na daty i wyrzucamy odwołane sesje.

**Dlaczego:** z tej tabeli wiemy, które sesje już się odbyły, i bierzemy z niej `session_key` / `meeting_key` do kolejnych zapytań. Tekstu nie da się porównać z „teraz”, daty — tak.

In [ ]:
sessions = get("sessions", year=2026)
for col in ["date_start", "date_end"]:
    sessions[col] = pd.to_datetime(sessions[col])
sessions = sessions[~sessions["is_cancelled"]]

sessions[sessions["country_name"] == "Singapore"][["session_key", "session_name", "date_start"]]

**Co tu się dzieje:**
- `pd.to_datetime` zamienia `"2026-10-11T12:00:00+00:00"` na datę ze strefą UTC.
- `~` to negacja — zostawiamy wiersze, gdzie `is_cancelled` jest fałszem.
- Ostatnie wyrażenie w komórce Jupyter wyświetla jako tabelę.

**Wniosek:** Singapur to **weekend sprinterski** (Practice 1 → Sprint Qualifying → Sprint → Qualifying → Race). Tylko jeden trening, ale za to dodatkowe dane ze sprintu. Wyścig: 12:00 UTC = **14:00 w Polsce**.

## 1.3 Wyścigi: zakończone + dzisiejszy

**Co robimy:** bierzemy wyścigi, które już wystartowały albo startują dziś, numerujemy je jako rundy i oznaczamy, które są zakończone.

**Dlaczego:** zakończone wyścigi to zbiór treningowy (~22 wiersze ze znanym wynikiem na wyścig). Dzisiejszy — Singapur — to ten, dla którego robimy predykcję: znamy jego kwalifikacje i sprint, ale nie wynik. Numer rundy przyda się do liczenia formy („ostatnie 3 wyścigi”).

In [ ]:
now = pd.Timestamp.now(tz="UTC")
races = (
    sessions[(sessions["session_name"] == "Race") & (sessions["date_start"] < now + pd.Timedelta(days=1))]
    .sort_values("date_start")
    .reset_index(drop=True)
)
races["round"] = races.index + 1
races["finished"] = races["date_end"] < now
races_done = races[races["finished"]]

TARGET_MEETING = races.loc[races["country_name"] == "Singapore", "meeting_key"].item()

print(len(races_done), "zakończonych wyścigów; cel predykcji: meeting_key =", TARGET_MEETING)
races[["round", "session_key", "meeting_key", "country_name", "location", "date_start", "finished"]]

**Co tu się dzieje:**
- `tz="UTC"` — pandas nie porówna daty ze strefą z datą bez strefy.
- Filtrujemy po `session_name == "Race"`, a nie `session_type`, bo **sprint też ma `session_type == "Race"`**.
- `now + 1 dzień` — dzięki temu łapiemy dzisiejszy wyścig, nawet jeśli jeszcze nie wystartował.
- `TARGET_MEETING` — klucz weekendu w Singapurze; `.item()` wyciąga pojedynczą wartość i rzuci błąd, gdyby pasowało 0 albo 2 wiersze (tania asercja).

**Do zauważenia w danych:**
1. GP Bahrajnu i Arabii Saudyjskiej w kwietniu są odwołane, a „Bahrain” wraca 4 października z `location` = **Kuala Lumpur**. Błąd w danych czy coś, czego nie wiemy? Lekcja: **zawsze oglądaj surowe dane**.
2. W `sessions` są też testy przedsezonowe („Day 1”…) — odpadły dzięki filtrowi na `"Race"`.

## 1.4 Kierowcy weekendu

**Co robimy:** funkcję, która dla danego weekendu (`meeting_key`) zwraca tabelę: numer → skrót nazwiska → zespół.

**Dlaczego:** wszystkie wyniki w OpenF1 znają kierowcę tylko po `driver_number`. A numer **nie jest stałym identyfikatorem** — w 2026 Norris jeździ z #1 (mistrz świata), a Verstappen z #3. Do łączenia danych między sezonami używamy `name_acronym` (VER, NOR…), a `team_name` jest potrzebny do porównań z partnerem z zespołu. Pytamy po `meeting_key` (1 zapytanie na weekend) zamiast po każdej sesji osobno — mniej zapytań, mniej problemów z limitem.

In [ ]:
DRIVER_COLS = ["driver_number", "name_acronym", "full_name", "team_name"]

def weekend_drivers(meeting_key):
    """Kierowcy weekendu: driver_number -> name_acronym, team_name."""
    drv = get("drivers", meeting_key=meeting_key)
    return drv[DRIVER_COLS].drop_duplicates("driver_number")

def session_key(meeting_key, session_name):
    """session_key konkretnej sesji weekendu albo None, jeśli takiej sesji nie było (np. sprint)."""
    s = sessions[(sessions["meeting_key"] == meeting_key) & (sessions["session_name"] == session_name)]
    return s["session_key"].item() if len(s) else None

weekend_drivers(TARGET_MEETING)

**Co tu się dzieje:**
- `drivers?meeting_key=...` zwraca kierowców dla **każdej sesji** weekendu, więc ten sam kierowca występuje kilka razy — `drop_duplicates` zostawia jeden wiersz na numer.
- `session_key()` to pomocnik: „daj mi klucz kwalifikacji z weekendu X”. Zwraca `None`, gdy sesji nie było — weekendy bez sprintu nie mają sesji `"Sprint"`.

## 1.5 Wyniki wyścigów

**Co robimy:** dla każdego zakończonego wyścigu pobieramy `session_result` i doklejamy kierowców.

**Dlaczego:** to nasze **zmienne docelowe** — z pozycji na mecie policzymy później „wygrał / top 3 / top 10”. A historia wyników posłuży też do cech formy.

In [ ]:
frames = []
for race in races_done.itertuples():
    res = get("session_result", session_key=race.session_key)
    df = res.merge(weekend_drivers(race.meeting_key), on="driver_number", how="left")
    df["round"] = race.round
    df["meeting_key"] = race.meeting_key
    frames.append(df)

results = pd.concat(frames, ignore_index=True)
print(results.shape)
results.head()

**Co tu się dzieje:**
- `itertuples()` iteruje po wierszach — `race.session_key` zamiast `race["session_key"]`.
- `merge(..., how="left")` — zostawia **wszystkie** wiersze wyników, nawet gdyby kierowcy brakowało w `drivers` (wtedy `name_acronym` = NaN). Dzięki temu taki problem zobaczymy, zamiast po cichu zgubić wiersz.
- `pd.concat` skleja 16 małych tabel w jedną.